# Generated Dialog Evaluation with Multiple LLMs

This notebook evaluates generated dialogs from video action annotations using three different LLMs via the OpenRouter API. The evaluation scores multiple criteria and averages results across the models for robust assessment.

## Import Required Libraries

In [1]:
import os
import json
import re
import time
from typing import List, Dict, Any, Tuple
from tqdm import tqdm
import numpy as np
# Import OpenRouter utilities
from mmassist.datasets.generate.openrouter_utils import OpenRouterGenerator

print("Libraries imported successfully!")

Libraries imported successfully!


## Configure OpenRouter API with Multiple Models

In [ ]:
# os.environ["OPENROUTER_API_KEY"] = "asdf"


In [14]:
# Configure three different LLM models for evaluation
# Make sure to set your OPENROUTER_API_KEY environment variable

MODEL_CONFIGS = {
    "claude": "anthropic/claude-sonnet-4",
    "openai": "openai/gpt-5", 
    "gemini": "google/gemini-2.5-flash"
}

# Initialize OpenRouter generators for each model
evaluators = {}
for name, model_id in MODEL_CONFIGS.items():
    try:
        evaluators[name] = OpenRouterGenerator.build(model_id=model_id)
        print(f"✓ Initialized {name} ({model_id})")
    except Exception as e:
        print(f"✗ Failed to initialize {name}: {e}")

print(f"\nInitialized {len(evaluators)} LLM evaluators")

✓ Initialized claude (anthropic/claude-sonnet-4)
✓ Initialized openai (openai/gpt-5)
✓ Initialized gemini (google/gemini-2.5-flash)

Initialized 3 LLM evaluators


## Load Generated Dialogs Data

In [15]:
# Configure data paths
DATA_ROOT_DIR = "/projects/beto/proassist_data"
# DIALOG_FILE_PATH = f"{DATA_ROOT_DIR}/processed_data/epfl/generated_dialogs/train/train_YH2014_2024_01_22_10_05_24.json"
DIALOG_FILE_PATH = f"{DATA_ROOT_DIR}/processed_data/epfl/generated_dialogs/train.json"

# Load generated dialogs
try:
    with open(DIALOG_FILE_PATH, "r") as f:
        generated_dialogs = json.load(f)
    
    print(f"Loaded {len(generated_dialogs)} generated dialogs")
    print(f"Sample dialog structure:")
    if generated_dialogs:
        print(json.dumps(generated_dialogs[0], indent=2)[:500] + "...")
        
except FileNotFoundError:
    print(f"Dialog file not found at {DIALOG_FILE_PATH}")
    print("Please update the path to your generated dialogs file")
    generated_dialogs = []

Loaded 35 generated dialogs
Sample dialog structure:
{
  "video_uid": "train_YH2018_2023_07_26_09_10_53",
  "inferred_goal": "Cooking Pad Thai",
  "inferred_knowledge": "Cooking Pad Thai\n1. Gather ingredients including shallots, tofu, eggs, shrimp, noodles, tamarind paste, and seasonings.\n2. Prepare the cooking area by setting up a cutting board, knife, and pans.\n3. Peel and cut shallots.\n4. Heat frying oil in a pan and cook shallots.\n5. Cut tofu into pieces and add to the pan, cooking until browned.\n6. Crack eggs into a bowl and whisk. Add ...


## Define Evaluation Prompts and Criteria

In [16]:
# System prompt for direct dialog evaluation (without reference comparison)
EVALUATION_SYS_PROMPT = """You are an expert in evaluating the quality of generated user-assistant dialogues. Your task is to evaluate dialog responses generated by a cooking assistant model from cooking video action annotations. The cooking assistant helps users complete tasks by providing guidance based on what it observes in the video.

Evaluation Criteria:
1. **Coherence**: How logically consistent and well-structured is the dialog? Do the assistant's responses follow naturally from the context?

2. **Relevance**: How well do the assistant's instructions relate to user needs? Are the suggestions appropriate for the situation?

3. **Naturalness**: How natural and human-like is the conversation flow? Does it feel like a realistic human-assistant interaction?

4. **Task Completion**: How effectively does the assistant guide the user toward completing their task? Are the instructions clear and actionable?

5. **Overall Quality**: The overall helpfulness and quality of the assistant's performance in the dialog.

Scoring Scale (1-5 for each criterion):
- 1 = Very Poor: Significant issues that make the dialog unhelpful or confusing
- 2 = Poor: Notable problems that detract from the dialog's usefulness  
- 3 = Average: Acceptable quality with some strengths and weaknesses
- 4 = Good: High quality with minor issues
- 5 = Excellent: Outstanding quality that would be very helpful to users

Instructions:
- Analyze the dialog carefully, focusing on the assistant's contributions
- Consider the context of video-based task assistance
- Provide a brief analysis followed by numerical scores
- Be consistent and objective in your evaluation"""

DIALOG_EVALUATION_PROMPT_TEMPLATE = """Generated dialog to evaluate:
{generated_dialog}

Please evaluate this dialog based on the five criteria: coherence, relevance, naturalness, task_completion, and overall quality.

Format your response as:
<brief analysis of the dialog's strengths and weaknesses>
---
{{"coherence": x, "relevance": y, "naturalness": z, "task_completion": w, "overall": v}}"""

# Define score fields for processing
SCORE_FIELDS = ["coherence", "relevance", "naturalness", "task_completion", "overall"]

print("Evaluation prompts and criteria defined!")

Evaluation prompts and criteria defined!


## Create Score Parsing Functions

In [17]:
def parse_scores(text: str) -> Dict[str, int] | None:
    """Parse scores from LLM response with robust error handling"""
    if not text:
        return None
        
    # Extract JSON portion after '---' separator
    if "---" in text:
        text = text.split("---", 1)[1]
    
    # Clean up the text
    text = text.strip()
    
    try:
        # Try to find JSON object in the text
        json_match = re.search(r'\{[^}]*\}', text)
        if json_match:
            json_str = json_match.group()
            scores = json.loads(json_str)
            
            # Validate that all expected fields are present and numeric
            if all(field in scores and isinstance(scores[field], (int, float)) 
                   for field in SCORE_FIELDS):
                return {field: int(scores[field]) for field in SCORE_FIELDS}
    
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        pass
    
    # If JSON parsing fails, try to extract numbers using regex
    try:
        score_dict = {}
        for field in SCORE_FIELDS:
            # Look for patterns like "field": 4 or "field": x
            pattern = rf'"{field}"\s*:\s*(\d+)'
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                score_dict[field] = int(match.group(1))
        
        if len(score_dict) == len(SCORE_FIELDS):
            return score_dict
            
    except ValueError:
        pass
    
    return None

def calculate_average_scores(all_scores: List[Dict[str, int]]) -> Dict[str, float]:
    """Calculate average scores across multiple evaluations"""
    if not all_scores:
        return {field: 0.0 for field in SCORE_FIELDS}
    
    avg_scores = {}
    for field in SCORE_FIELDS:
        scores = [s[field] for s in all_scores if field in s]
        avg_scores[field] = sum(scores) / len(scores) if scores else 0.0
    
    return avg_scores

def format_dialog_for_evaluation(dialog_data: Dict[str, Any]) -> Tuple[str, str]:
    """Convert dialog data to formatted strings for evaluation"""
    # Extract dialog text from the conversation structure
    dialog_text = ""
    action_context = ""
    
    # Extract conversation from the conversations array
    if "conversations" in dialog_data and len(dialog_data["conversations"]) > 0:
        conversation_turns = dialog_data["conversations"][0].get("conversation", [])
        
        # Format conversation turns
        for turn in conversation_turns:
            role = turn.get("role", "unknown")
            content = turn.get("content", "")
            time_stamp = turn.get("time", "")
            
            # Include timestamp for context
            if time_stamp:
                dialog_text += f"{role} (t={time_stamp}s): {content}\n"
            else:
                dialog_text += f"{role}: {content}\n"
    
    # Extract inferred knowledge as action context if available
    if "inferred_knowledge" in dialog_data:
        action_context = dialog_data["inferred_knowledge"]
    elif "inferred_goal" in dialog_data:
        action_context = f"Goal: {dialog_data['inferred_goal']}"
    
    return dialog_text.strip(), action_context.strip()

print("Score parsing and utility functions defined!")

Score parsing and utility functions defined!


## Process Dialogs and Create Evaluation Prompts

In [18]:
res = format_dialog_for_evaluation(generated_dialogs[0])
res[0]

"user: I'm going to make Pad Thai today!\nassistant: Great! Pad Thai is a delicious choice. To start, please read through the recipe to familiarize yourself with the steps.\nassistant (t=45.0s): Now that you've read the recipe, let's gather our ingredients. First, open the fridge and grab the package of tofu, the box of rice noodles, and the package of bean sprouts.\nassistant (t=65.2s): Excellent. Next, close the fridge and open the cupboard. We're looking for the shallots, fish sauce, tamarind paste, and soy sauce.\nassistant (t=137.4s): Perfect! Close the cupboard once you have those.\nassistant (t=145.2s): Take a moment to review the recipe again, and then wash your hands before we move on.\nassistant (t=150.4s): Now, let's get some cooking tools ready. Grab a cutting board, a knife, and two pans from the cupboard.\nassistant (t=173.3s): Great job! Place one pan on the stove.\nassistant (t=176.4s): Now, grab the frying oil and pour some into the pan.\nassistant (t=199.9s): Once you

In [22]:
# Prepare evaluation data
evaluation_data = []
evaluation_prompts = []

# Limit the number of dialogs to evaluate (for testing/cost control)
MAX_DIALOGS = min(3, len(generated_dialogs))  # Adjust as needed

print(f"Preparing evaluation for {MAX_DIALOGS} dialogs...")

for i, dialog_data in enumerate(generated_dialogs[:MAX_DIALOGS]):
    # Format dialog and action context
    dialog_text, action_context = format_dialog_for_evaluation(dialog_data)
    
    if not dialog_text:
        print(f"Skipping dialog {i}: No dialog text found")
        continue
    
    # Create evaluation prompt
    user_prompt = DIALOG_EVALUATION_PROMPT_TEMPLATE.format(
        generated_dialog=dialog_text,
        # action_context=action_context if action_context else "No action context available"
    )
    
    # Store for evaluation
    evaluation_data.append({
        "id": i,
        "dialog_text": dialog_text,
        # "action_context": action_context,
        "original_data": dialog_data
    })
    
    # Create prompt for each LLM
    prompt = [("system", EVALUATION_SYS_PROMPT), ("user", user_prompt)]
    evaluation_prompts.append(prompt)

print(f"Prepared {len(evaluation_prompts)} evaluation prompts")

# Show a sample prompt
if evaluation_prompts:
    print("\nSample evaluation prompt:")
    print("="*50)
    print("SYSTEM:", evaluation_prompts[0][0][1][:200] + "...")
    print("USER:", evaluation_prompts[0][1][1][:300] + "...")
    print("="*50)

Preparing evaluation for 3 dialogs...
Prepared 3 evaluation prompts

Sample evaluation prompt:
SYSTEM: You are an expert in evaluating the quality of generated user-assistant dialogues. Your task is to evaluate dialog responses generated by a cooking assistant model from cooking video action annotation...
USER: Generated dialog to evaluate:
user: I'm going to make Pad Thai today!
assistant: Great! Pad Thai is a delicious choice. To start, please read through the recipe to familiarize yourself with the steps.
assistant (t=45.0s): Now that you've read the recipe, let's gather our ingredients. First, open the...


In [27]:
len(evaluation_prompts)

3

## Run Multi-Model LLM Evaluation

In [23]:
# Run evaluation with all models
model_results = {}
batch_size = 5  # Smaller batch size for API rate limiting

for model_name, evaluator in evaluators.items():
    print(f"\n{'='*50}")
    print(f"Running evaluation with {model_name.upper()}")
    print(f"{'='*50}")
    
    model_outputs = []
    
    # Process in batches to avoid rate limiting
    for i in tqdm(range(0, len(evaluation_prompts), batch_size), 
                  desc=f"Evaluating with {model_name}"):
        batch_prompts = evaluation_prompts[i:i+batch_size]
        
        try:
            # Use batch_generate for efficiency
            batch_outputs = evaluator.batch_generate(
                batch_prompts, 
                temperature=0.3,  # Lower temperature for more consistent scoring
                max_tokens=1024
            )
            
            # batch_generate returns List[List[str]], we need List[str]
            for output_list in batch_outputs:
                model_outputs.append(output_list[0])  # Take first (and only) output
        
        except Exception as e:
            print(f"Error in batch {i//batch_size + 1}: {e}")
            # Add empty outputs for failed batch
            for _ in range(len(batch_prompts)):
                model_outputs.append("")
        
        # Add delay between batches to respect rate limits
        if i + batch_size < len(evaluation_prompts):
            time.sleep(2)
    
    model_results[model_name] = model_outputs
    print(f"Completed evaluation with {model_name}: {len(model_outputs)} responses")

print(f"\nCompleted evaluation with {len(model_results)} models")
print("Response counts:", {name: len(outputs) for name, outputs in model_results.items()})


Running evaluation with CLAUDE


Evaluating with claude:   0%|          | 0/1 [00:00<?, ?it/s]

Processing conversation 1/3
Processing conversation 2/3
Processing conversation 3/3


Evaluating with claude: 100%|██████████| 1/1 [00:33<00:00, 33.94s/it]


Completed evaluation with claude: 3 responses

Running evaluation with OPENAI


...

Completed evaluation with openai: 3 responses

Running evaluation with GEMINI


Evaluating with gemini:   0%|          | 0/1 [00:00<?, ?it/s]

Processing conversation 1/3
Processing conversation 2/3
Processing conversation 3/3


Evaluating with gemini: 100%|██████████| 1/1 [00:12<00:00, 12.07s/it]

Completed evaluation with gemini: 3 responses

Completed evaluation with 3 models
Response counts: {'claude': 3, 'openai': 3, 'gemini': 3}


## Parse and Average Scores Across Models

In [25]:
# Parse scores from all models
parsed_results = {}
parsing_stats = {}

for model_name, outputs in model_results.items():
    print(f"\nParsing scores from {model_name}...")
    
    parsed_scores = []
    parse_failures = 0
    
    for i, output in enumerate(outputs):
        scores = parse_scores(output)
        if scores:
            parsed_scores.append(scores)
        else:
            parse_failures += 1
            # Print first few failures for debugging
            if parse_failures <= 3:
                print(f"Parse failure {parse_failures} for {model_name}:")
                print(f"Output: {output[:200]}...")
                print("-" * 30)
    
    parsed_results[model_name] = parsed_scores
    parsing_stats[model_name] = {
        "total_responses": len(outputs),
        "successful_parses": len(parsed_scores),
        "parse_failures": parse_failures,
        "success_rate": len(parsed_scores) / len(outputs) if outputs else 0
    }
    
    print(f"{model_name} parsing stats: {parsing_stats[model_name]}")

# Combine scores across models for each dialog
final_results = []

for i in range(len(evaluation_data)):
    dialog_result = evaluation_data[i].copy()
    
    # Collect scores from all models for this dialog
    model_scores = {}
    all_valid_scores = []
    
    for model_name in evaluators.keys():
        if i < len(parsed_results.get(model_name, [])):
            scores = parsed_results[model_name][i]
            model_scores[f"{model_name}_scores"] = scores
            all_valid_scores.append(scores)
        else:
            model_scores[f"{model_name}_scores"] = None
    
    # Calculate average scores across models
    if all_valid_scores:
        avg_scores = calculate_average_scores(all_valid_scores)
        dialog_result.update({
            "model_scores": model_scores,
            "average_scores": avg_scores,
            "num_valid_evaluations": len(all_valid_scores)
        })
    else:
        dialog_result.update({
            "model_scores": model_scores,
            "average_scores": {field: None for field in SCORE_FIELDS},
            "num_valid_evaluations": 0
        })
    
    final_results.append(dialog_result)

print(f"\nProcessed {len(final_results)} dialog evaluations")
print(f"Parsing success rates: {parsing_stats}")


Parsing scores from claude...
claude parsing stats: {'total_responses': 3, 'successful_parses': 3, 'parse_failures': 0, 'success_rate': 1.0}

Parsing scores from openai...
Parse failure 1 for openai:
Output: ...
------------------------------
openai parsing stats: {'total_responses': 3, 'successful_parses': 2, 'parse_failures': 1, 'success_rate': 0.6666666666666666}

Parsing scores from gemini...
gemini parsing stats: {'total_responses': 3, 'successful_parses': 3, 'parse_failures': 0, 'success_rate': 1.0}

Processed 3 dialog evaluations
Parsing success rates: {'claude': {'total_responses': 3, 'successful_parses': 3, 'parse_failures': 0, 'success_rate': 1.0}, 'openai': {'total_responses': 3, 'successful_parses': 2, 'parse_failures': 1, 'success_rate': 0.6666666666666666}, 'gemini': {'total_responses': 3, 'successful_parses': 3, 'parse_failures': 0, 'success_rate': 1.0}}


## Calculate Final Metrics and Save Results

In [26]:
# Calculate overall statistics
overall_stats = {}

# Get all dialogs with valid average scores
valid_results = [r for r in final_results if r["num_valid_evaluations"] > 0]

print(f"Statistical Summary ({len(valid_results)} valid evaluations):")
print("="*60)

for field in SCORE_FIELDS:
    scores = [r["average_scores"][field] for r in valid_results 
              if r["average_scores"][field] is not None]
    
    if scores:
        overall_stats[field] = {
            "mean": np.mean(scores),
            "std": np.std(scores),
            "min": np.min(scores),
            "max": np.max(scores),
            "count": len(scores)
        }
        
        print(f"{field.upper()}")
        print(f"  Mean: {overall_stats[field]['mean']:.2f} ± {overall_stats[field]['std']:.2f}")
        print(f"  Range: {overall_stats[field]['min']:.1f} - {overall_stats[field]['max']:.1f}")
        print()
    else:
        overall_stats[field] = None
        print(f"{field.upper()}: No valid scores")

# Model comparison
print("\nModel Comparison:")
print("="*40)
for model_name in evaluators.keys():
    stats = parsing_stats[model_name]
    print(f"{model_name.upper()}: {stats['successful_parses']}/{stats['total_responses']} "
          f"({stats['success_rate']:.1%} success rate)")

# Save detailed results
timestamp = time.strftime("%Y%m%d_%H%M%S")
results_file = f"dialog_evaluation_results_{timestamp}.json"
summary_file = f"dialog_evaluation_summary_{timestamp}.json"

# Save detailed results
save_data = {
    "metadata": {
        "timestamp": timestamp,
        "num_dialogs_evaluated": len(valid_results),
        "models_used": list(evaluators.keys()),
        "evaluation_criteria": SCORE_FIELDS
    },
    "parsing_stats": parsing_stats,
    "overall_statistics": overall_stats,
    "detailed_results": final_results
}

with open(results_file, "w") as f:
    json.dump(save_data, f, indent=2)

# Save summary only
summary_data = {
    "metadata": save_data["metadata"],
    "parsing_stats": parsing_stats,
    "overall_statistics": overall_stats
}

with open(summary_file, "w") as f:
    json.dump(summary_data, f, indent=2)

print(f"\nResults saved:")
print(f"  Detailed: {results_file}")
print(f"  Summary:  {summary_file}")

# Display top and bottom performers
if valid_results:
    print(f"\nTop 3 Performers (by overall score):")
    sorted_results = sorted(valid_results, 
                          key=lambda x: x["average_scores"]["overall"] or 0, 
                          reverse=True)
    
    for i, result in enumerate(sorted_results[:3]):
        overall_score = result["average_scores"]["overall"]
        print(f"  {i+1}. Dialog {result['id']}: {overall_score:.2f} overall")
    
    print(f"\nBottom 3 Performers (by overall score):")
    for i, result in enumerate(sorted_results[-3:]):
        overall_score = result["average_scores"]["overall"]
        print(f"  {len(valid_results)-2+i}. Dialog {result['id']}: {overall_score:.2f} overall")

Statistical Summary (3 valid evaluations):
COHERENCE
  Mean: 2.00 ± 0.27
  Range: 1.7 - 2.3

RELEVANCE
  Mean: 2.56 ± 0.42
  Range: 2.0 - 3.0

NATURALNESS
  Mean: 1.33 ± 0.27
  Range: 1.0 - 1.7

TASK_COMPLETION
  Mean: 2.61 ± 0.28
  Range: 2.3 - 3.0

OVERALL
  Mean: 2.00 ± 0.27
  Range: 1.7 - 2.3


Model Comparison:
CLAUDE: 3/3 (100.0% success rate)
OPENAI: 2/3 (66.7% success rate)
GEMINI: 3/3 (100.0% success rate)

Results saved:
  Detailed: dialog_evaluation_results_20250926_083805.json
  Summary:  dialog_evaluation_summary_20250926_083805.json

Top 3 Performers (by overall score):
  1. Dialog 0: 2.33 overall
  2. Dialog 2: 2.00 overall
  3. Dialog 1: 1.67 overall

Bottom 3 Performers (by overall score):
  1. Dialog 0: 2.33 overall
  2. Dialog 2: 2.00 overall
  3. Dialog 1: 1.67 overall


## Sample Evaluation Results

Let's examine some specific examples to understand the evaluation results.

In [12]:
# Display sample evaluation results
if valid_results:
    print("SAMPLE EVALUATION RESULTS")
    print("=" * 80)
    
    # Show a high-scoring dialog
    high_scorer = max(valid_results, key=lambda x: x["average_scores"]["overall"] or 0)
    print(f"\n🏆 HIGHEST SCORING DIALOG (ID: {high_scorer['id']}):")
    print(f"Overall Score: {high_scorer['average_scores']['overall']:.2f}")
    print(f"Dialog Text: {high_scorer['dialog_text'][:300]}...")
    print("\nScores by Model:")
    for model_name in evaluators.keys():
        model_scores = high_scorer['model_scores'].get(f"{model_name}_scores")
        if model_scores:
            print(f"  {model_name}: {model_scores}")
    print(f"Average: {high_scorer['average_scores']}")
    
    # Show a low-scoring dialog
    low_scorer = min(valid_results, key=lambda x: x["average_scores"]["overall"] or 5)
    print(f"\n📉 LOWEST SCORING DIALOG (ID: {low_scorer['id']}):")
    print(f"Overall Score: {low_scorer['average_scores']['overall']:.2f}")
    print(f"Dialog Text: {low_scorer['dialog_text'][:300]}...")
    print("\nScores by Model:")
    for model_name in evaluators.keys():
        model_scores = low_scorer['model_scores'].get(f"{model_name}_scores")
        if model_scores:
            print(f"  {model_name}: {model_scores}")
    print(f"Average: {low_scorer['average_scores']}")
    
    # Show model agreement analysis
    print(f"\n📊 MODEL AGREEMENT ANALYSIS:")
    agreements = []
    for result in valid_results:
        model_overall_scores = []
        for model_name in evaluators.keys():
            scores = result['model_scores'].get(f"{model_name}_scores")
            if scores and 'overall' in scores:
                model_overall_scores.append(scores['overall'])
        
        if len(model_overall_scores) >= 2:
            # Calculate standard deviation as a measure of agreement
            agreements.append(np.std(model_overall_scores))
    
    if agreements:
        avg_disagreement = np.mean(agreements)
        print(f"Average model disagreement (std dev): {avg_disagreement:.2f}")
        print(f"High agreement (std < 0.5): {sum(1 for a in agreements if a < 0.5)} dialogs")
        print(f"Low agreement (std > 1.0): {sum(1 for a in agreements if a > 1.0)} dialogs")

else:
    print("No valid evaluation results to display.")

No valid evaluation results to display.
